In [1]:
import torch as pt
import numpy as np
from torch.utils.data import DataLoader
from utils.data import SpecDataset, collate_fun_emb
from utils.tools import gen_embeddings, build_idx, evaluate

In [ ]:
mols_test = pt.load('./data/mine/test_11499.pt')
mols_all = pt.load('./data/mine/mols_all.pt')
dataset_lib = SpecDataset(mols_all)
loader_lib = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_test = SpecDataset(mols_test)
loader_test = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)

In [8]:
from copy import deepcopy


mols_test_by_interval = []
for i in range(1, 11):
    mols_test_i = deepcopy(mols_test)
    for spec in mols_test_i:
        intensities = spec.intensities
        mask = intensities < 0.1 * i
        intensities[mask] = np.sqrt(intensities[mask])
        spec._peaks._intensities = intensities
    mols_test_by_interval.append(mols_test_i)

In [9]:
mols_all_by_interval = []
for i in range(1, 11):
    mols_all_i = deepcopy(mols_all)
    for spec in mols_all_i:
        intensities = spec.intensities
        mask = intensities < 0.1 * i
        intensities[mask] = np.sqrt(intensities[mask])
        spec._peaks._intensities = intensities
    mols_all_by_interval.append(mols_all_i)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Spec2Emb(nn.Module):
    def __init__(self, num_emb:int=1000, emb_dim:int=500):
        super(Spec2Emb, self).__init__()
        self.max_exp = 6
        self.emb_con = nn.Embedding(
            num_embeddings=num_emb,
            embedding_dim=emb_dim,
        )
        self.emb_cen = nn.Embedding(
            num_embeddings=num_emb,
            embedding_dim=emb_dim,
        )
        self.trip_loss = nn.TripletMarginLoss(margin=1.0, p=2)

    def _compute_embedding(self, data, power):
        mzs, intens, masks = data  # [batch, seq]
        embs = self.emb_cen(mzs) # [batch, seq, emb_dim]
        embs = embs * masks.unsqueeze(-1)
        # intens = pt.pow(intens, power) # [batch, seq]
        embs = (embs * intens.unsqueeze(-1)).sum(dim=1) # [batch, emb_dim]
        return embs

    def forward(self, data, mode:str='train', power:float=0.5):
        if mode == 'train': 
            mzs_con, masks_con, poss_cen, batch_idx, negs_cen, masks_neg = data
            embs_con = self.emb_con(mzs_con)        # [batch, seq, emb_dim]
            embs_pos = self.emb_cen(poss_cen)     # [B, emb_dim]
            embs_neg = self.emb_cen(negs_cen)      # [B, neg_num, emb_dim]
            embs_neg *= masks_neg.unsqueeze(-1)
            # for every cen word its context words
            embs_con = embs_con[batch_idx] * masks_con.unsqueeze(-1)
            embs_con = embs_con.sum(dim=1) / masks_con.sum(dim=1).unsqueeze(-1) # [B, emb_dim]
            pos_score = (embs_con * embs_pos).sum(dim=-1) # 点积
            pos_score = pt.clamp(pos_score, max=self.max_exp, min=-self.max_exp)
            pos_score = -F.logsigmoid(pos_score)
            neg_score = pt.bmm(embs_neg, embs_con.unsqueeze(-1)).squeeze(-1) # 
            neg_score = pt.clamp(neg_score, max=self.max_exp, min=-self.max_exp)
            neg_score = -F.logsigmoid(-neg_score).sum(dim=-1)
            return (pos_score + neg_score).sum() 
        elif mode == 'emb':            
            return self._compute_embedding(data, power)
        elif mode == 'finetune':
            data_mea, data_pre_hit, data_pre_nhit = data
            embs_mea = self._compute_embedding(data_mea, power)
            embs_pre_hit = self._compute_embedding(data_pre_hit, power)
            embs_pre_nhit = self._compute_embedding(data_pre_nhit, power)
            # batchsize, emb_dim
            embs_mea = F.normalize(embs_mea, p=2, dim=-1)
            embs_pre_hit = F.normalize(embs_pre_hit, p=2, dim=-1)
            embs_pre_nhit = F.normalize(embs_pre_nhit, p=2, dim=-1)
            # batchsize
            loss = self.trip_loss(embs_mea, embs_pre_hit, embs_pre_nhit)
            return loss
        else:
            raise ValueError('mode not exist')

In [11]:
gpu = 6
model = Spec2Emb().to(gpu)
model.load_state_dict(pt.load('./model/base_peak0.01_epoch4.pth', map_location='cpu'))

/tmp/ipykernel_4124470/3060180341.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(pt.load('./model/base_peak0.01_epoch4.pth', map_location='cpu'))


<All keys matched successfully>

In [12]:
from utils.tools import gen_embeddings, build_idx, evaluate


embeddings_lib_by_interval = []
embeddings_test_by_interval = []
for i in range(10):
    print(f'Processing {i}th embedding...')
    dataset_lib_i = SpecDataset(mols_all_by_interval[i])
    loader_lib_i = DataLoader(dataset_lib_i, batch_size=2048, shuffle=False,
                              num_workers=8, collate_fn=collate_fun_emb)
    dataset_test_i = SpecDataset(mols_test_by_interval[i])
    loader_test_i = DataLoader(dataset_test_i, batch_size=2048, shuffle=False,
                               num_workers=8, collate_fn=collate_fun_emb)
    embeddings_lib_i = gen_embeddings(model, loader_lib_i, gpu, power=1.0)
    embeddings_test_i = gen_embeddings(model, loader_test_i, gpu, power=1.0)
    embeddings_lib_by_interval.append(embeddings_lib_i)
    embeddings_test_by_interval.append(embeddings_test_i)

Processing 0th embedding...


Processing 1th embedding...
Processing 2th embedding...
Processing 3th embedding...
Processing 4th embedding...
Processing 5th embedding...
Processing 6th embedding...
Processing 7th embedding...
Processing 8th embedding...
Processing 9th embedding...


In [13]:
file = open('./test_by_interval.txt', 'w')

In [17]:
I_by_interval = []
D_insilico_by_interval = []
for i in range(10):
    I_i, D_i = build_idx(embeddings_lib_by_interval[i][:2146690], 
                        embeddings_test_by_interval[i], gpu)
    top1_i, top10_i = evaluate(mols_test_by_interval[i], I_i, 
                               mols_all_by_interval[i][:2146690], f=file)
file.close()

Searching time:  0:00:01.496862
insilico library
Top1 hit rate: 31.23%
Top10 hit rate: 71.10%
Searching time:  0:00:01.489932
insilico library
Top1 hit rate: 32.69%
Top10 hit rate: 73.93%
Searching time:  0:00:01.480815
insilico library
Top1 hit rate: 34.79%
Top10 hit rate: 75.82%
Searching time:  0:00:01.491389
insilico library
Top1 hit rate: 36.06%
Top10 hit rate: 77.64%
Searching time:  0:00:01.495853
insilico library
Top1 hit rate: 37.47%
Top10 hit rate: 79.25%
Searching time:  0:00:01.477680
insilico library
Top1 hit rate: 38.31%
Top10 hit rate: 79.95%
Searching time:  0:00:01.482883
insilico library
Top1 hit rate: 38.79%
Top10 hit rate: 80.51%
Searching time:  0:00:01.480593
insilico library
Top1 hit rate: 39.22%
Top10 hit rate: 80.72%
Searching time:  0:00:01.504697
insilico library
Top1 hit rate: 39.51%
Top10 hit rate: 80.83%
Searching time:  0:00:01.487017
insilico library
Top1 hit rate: 39.58%
Top10 hit rate: 80.86%
